# Adult dataset operating modes

This notebook demonstrates MIMIC's main operating modes on the Adult/Census Income dataset.

Set `N_ROWS` to control the working sample size. The default is intentionally small enough for quick experimentation.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA

from mimic import MIMIC, GenerationPolicy, RandomForestPathEncoder, MixedFeatureDecoder

RANDOM_STATE = 42
N_ROWS = 1000
rng = np.random.default_rng(RANDOM_STATE)

## Load and clean Adult

Adult has no real ID column, but it does contain sampling-weight and education-code columns that are not useful for this demonstration. `fnlwgt` has a very wide range and `education-num` duplicates information already present in `education`, so both are excluded.

In [ ]:
adult = fetch_openml("adult", version=2, as_frame=True)
raw = adult.frame.copy()
raw = raw.replace("?", np.nan)

drop_columns = ["fnlwgt", "education-num", "capital-gain", "capital-loss"]
df = raw.drop(columns=drop_columns).dropna().sample(n=N_ROWS, random_state=RANDOM_STATE).reset_index(drop=True)
df["income"] = df["class"].astype(str)
df = df.drop(columns=["class"])

regression_columns = ["age",  "hours-per-week"]
classification_columns = [c for c in df.columns if c not in regression_columns]

df.head(), df.shape, df["income"].value_counts(normalize=True)

(   age         workclass     education         marital-status  \
 0   19  Self-emp-not-inc          10th  Married-spouse-absent   
 1   45           Private       HS-grad     Married-civ-spouse   
 2   47           Private     Assoc-voc     Married-civ-spouse   
 3   23           Private  Some-college          Never-married   
 4   53         Local-gov       HS-grad     Married-civ-spouse   
 
         occupation    relationship                race     sex  capital-gain  \
 0     Adm-clerical       Unmarried  Amer-Indian-Eskimo  Female             0   
 1  Farming-fishing         Husband  Amer-Indian-Eskimo    Male             0   
 2     Craft-repair         Husband               White    Male             0   
 3     Craft-repair  Other-relative               White    Male             0   
 4     Craft-repair         Husband               White    Male             0   
 
    capital-loss  hours-per-week native-country income  
 0             0              40  United-States  <=50K  


## Fit from clean data

The model is fitted on clean complete rows. Later cells create perturbed copies to test each operating mode.

In [ ]:
mimic = MIMIC(
    regression_columns=regression_columns,
    classification_columns=classification_columns,
    encoder=RandomForestPathEncoder(n_estimators=25, embedding_dim=10, random_state=RANDOM_STATE),
    decoder=MixedFeatureDecoder.random_forest(n_estimators=25, random_state=RANDOM_STATE),
    policy=GenerationPolicy(method="smote", neighbour_mode="normal", n_neighbors=5, lambda_range=(0.0, 1.0)),
    n_bootstrap=2,
    random_state=RANDOM_STATE,
)
mimic.fit(df)
embedding = mimic.transform(df)
embedding.shape

## 1. Missing-value imputation

Hide known values and measure how well MIMIC reconstructs them.

In [ ]:
impute_columns = ["age", "hours-per-week", "workclass", "marital-status"]
masked = df.copy()
truth = []
for col in impute_columns:
    rows = rng.choice(masked.index, size=max(20, N_ROWS // 20), replace=False)
    truth.append(pd.DataFrame({"row": rows, "column": col, "true": df.loc[rows, col].to_numpy()}))
    masked.loc[rows, col] = np.nan
truth = pd.concat(truth, ignore_index=True)

reconstructed = mimic.impute(masked, columns=impute_columns)
truth["predicted"] = [reconstructed.loc[r, c] for r, c in zip(truth["row"], truth["column"])]

imputation_scores = []
for col in impute_columns:
    part = truth[truth["column"] == col]
    if col in regression_columns:
        score = mean_absolute_error(part["true"].astype(float), part["predicted"].astype(float))
        metric = "MAE"
    else:
        score = accuracy_score(part["true"].astype(str), part["predicted"].astype(str))
        metric = "accuracy"
    imputation_scores.append({"column": col, "metric": metric, "score": score})

pd.DataFrame(imputation_scores), truth.head()

## 2. Data inconsistency check

Inject unrealistic numerical values beyond two standard deviations from the clean distribution and rank entries by prediction discrepancy.

In [ ]:
corrupted = df.copy()
for col in ["age", "hours-per-week"]:
    corrupted[col] = corrupted[col].astype(float)

corruption_records = []
for col in ["age", "hours-per-week"]:
    rows = rng.choice(corrupted.index, size=max(10, N_ROWS // 50), replace=False)
    mu = df[col].mean()
    sigma = df[col].std()
    corrupted.loc[rows, col] = mu + 4 * sigma
    corruption_records.extend({"row_index": r, "column": col} for r in rows)
corruption_records = pd.DataFrame(corruption_records)

diagnostics = mimic.confidence(corrupted, columns=["age", "hours-per-week"])
merged_marker = diagnostics.merge(
    corruption_records.assign(was_corrupted=True),
    on=["row_index", "column"],
    how="left",
)["was_corrupted"]
diagnostics["was_corrupted"] = (merged_marker == True).to_numpy()

ranked = diagnostics.sort_values("discrepancy", ascending=False)
top_k = len(corruption_records)
precision_at_k = ranked.head(top_k)["was_corrupted"].mean()
precision_at_k, ranked[["row_index", "column", "observed", "prediction", "discrepancy", "was_corrupted"]].head(12)

## 3. Supervised learning on income

Income prediction is represented as imputing a missing `income` target column.

In [ ]:
test_rows = rng.choice(df.index, size=N_ROWS // 4, replace=False)
income_task = df.copy()
income_truth = df.loc[test_rows, "income"].copy()
income_task.loc[test_rows, "income"] = np.nan

income_pred = mimic.impute(income_task, columns=["income"])
y_true = income_truth.astype(str)
y_pred = income_pred.loc[test_rows, "income"].astype(str)

income_metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "macro_f1": f1_score(y_true, y_pred, average="macro"),
}
income_confidence = mimic.confidence(income_task.loc[test_rows], columns=["income"])
income_metrics, income_confidence.head()

## 4. Synthetic generation

Generate synthetic rows, inspect traceability, and compare original and generated distributions in a shared 2D projection.

In [ ]:
synthetic, trace = mimic.sample(300, return_trace=True)
synthetic.head(), trace.head()

In [ ]:
combined = pd.concat(
    [df.assign(source="original"), synthetic.assign(source="synthetic")],
    ignore_index=True,
)

# Use MIMIC's fitted embedding for both original and generated rows, then project jointly to 2D.
H_combined = mimic.transform(combined.drop(columns=["source"]))
coords = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(H_combined)
plot_df = pd.DataFrame(coords, columns=["pc1", "pc2"])
plot_df["source"] = combined["source"].to_numpy()

fig, ax = plt.subplots(figsize=(7, 5))
for source, marker, alpha in [("original", "o", 0.45), ("synthetic", "x", 0.75)]:
    part = plot_df[plot_df["source"] == source]
    ax.scatter(part["pc1"], part["pc2"], label=source, marker=marker, alpha=alpha, s=24)
ax.set_title("Original vs generated rows in MIMIC embedding PCA")
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.legend()
fig.tight_layout()

In [ ]:
summary = pd.concat(
    [df[regression_columns].describe().T.add_prefix("original_"), synthetic[regression_columns].describe().T.add_prefix("synthetic_")],
    axis=1,
)
summary[["original_mean", "synthetic_mean", "original_std", "synthetic_std"]]